<a href="https://colab.research.google.com/github/thanatchasan/Ge234/blob/main/Lab4_GE234_NumPy_GeoPandas_6606614714.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🌍 Lab 4: การประมวลผลข้อมูลด้วย NumPy และ GeoPandas
## วิชา GE 234 Basic Programming for Geographers

### 🎯 **วัตถุประสงค์**
1. เข้าใจการใช้ **NumPy** สำหรับการประมวลผลข้อมูลทางภูมิศาสตร์ เช่น ข้อมูล Raster และพิกัด
2. ใช้ **GeoPandas** ในการจัดการและวิเคราะห์ข้อมูลเวกเตอร์ เช่น **Shapefile**
3. สามารถดำเนินการทางสถิติกับข้อมูลพิกัดและชั้นข้อมูลทางภูมิศาสตร์ได้
4. สามารถใช้ NumPy และ GeoPandas ร่วมกันเพื่อวิเคราะห์ข้อมูลได้

---

## 🔹 แบบฝึกหัด 1: ใช้ NumPy คำนวณค่า Mean, Max, Min ของค่า NDVI ในอาร์เรย์ที่สร้างขึ้นเอง


In [2]:
import numpy as np

# สร้างข้อมูลตัวอย่างสำหรับภาพ NDVI (Normalized Difference Vegetation Index)
nir = np.array([[0.8, 0.9, 0.7], [0.6, 0.7, 0.4], [0.5, 0.8, 0.3]])
red = np.array([[0.4, 0.5, 0.3], [0.2, 0.4, 0.1], [0.1, 0.2, 0.1]])

# คำนวณค่า NDVI
ndvi = (nir - red) / (nir + red)

print("ค่า NDVI:")
print(ndvi)

ค่า NDVI:
[[0.33333333 0.28571429 0.4       ]
 [0.5        0.27272727 0.6       ]
 [0.66666667 0.6        0.5       ]]


In [ ]:
print(f"ค่าเฉลี่ย NDVI: {np.mean(ndvi):.2f}")
print(f"ค่า NDVI สูงสุด: {np.max(ndvi):.2f}")
print(f"ค่า NDVI ต่ำสุด: {np.min(ndvi):.2f}")

ค่าเฉลี่ย NDVI: 0.46
ค่า NDVI สูงสุด: 0.67
ค่า NDVI ต่ำสุด: 0.27



## 🔹 แบบฝึกหัด 2: ใช้ GeoPandas โหลด Shapefile ของจังหวัด และคำนวณพื้นที่ของแต่ละจังหวัด


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลจังหวัดของประเทศไทย
gdf = gpd.read_file("/content/drive/MyDrive/Colab Notebooks/ไืทย", layer='gadm41_THA_1')

# แสดงข้อมูล 5 แถวแรก
print(gdf.head())


     GID_1 GID_0   COUNTRY              NAME_1  \
0  THA.1_1   THA  Thailand       Amnat Charoen   
1  THA.2_1   THA  Thailand           Ang Thong   
2  THA.3_1   THA  Thailand  Bangkok Metropolis   
3  THA.4_1   THA  Thailand           Bueng Kan   
4  THA.5_1   THA  Thailand            Buri Ram   

                          VARNAME_1          NL_NAME_1    TYPE_1 ENGTYPE_1  \
0                                NA  จังหวัดอำนาจเจริญ  Changwat  Province   
1                                NA     จังหวัดอ่างทอง  Changwat  Province   
2  Bangkok|Krung Thep|Krung Thep Ma   จังหวัดเชียงใหม่  Changwat  Province   
3                                NA             บึงกาฬ  Changwat  Province   
4                          Buri Rum   จังหวัดบุรีรัมย์  Changwat  Province   

  CC_1 HASC_1  ISO_1                                           geometry  
0   37  TH.AC  TH-37  POLYGON ((104.58696 15.60588, 104.58676 15.605...  
1   15  TH.AT  TH-15  POLYGON ((100.38371 14.74216, 100.386 14.7427,...  
2   10  

In [5]:

# ตรวจสอบค่า CRS (Coordinate Reference System)
print(gdf.crs)

# แปลงระบบพิกัด (CRS) เป็น Projected CRS เพื่อการคำนวณพื้นที่ที่แม่นยำ (เช่น UTM Zone 47N สำหรับประเทศไทย)
gdf_projected = gdf.to_crs(epsg=32647)

# คำนวณพื้นที่ของแต่ละจังหวัด (หน่วยเป็นตารางเมตร) จาก gdf_projected
gdf["area_sqkm"] = gdf_projected.geometry.area / 1e6

# แสดงพื้นที่จังหวัด 5 อันดับแรกที่ใหญ่ที่สุด
print(gdf.nlargest(5, "area_sqkm")[["NAME_1", "area_sqkm"]])


EPSG:4326
           NAME_1    area_sqkm
831           Tak  4815.643185
242  Kanchanaburi  3850.424873
894   Uthai Thani  3624.977978
151    Chiang Mai  3355.125626
238  Kanchanaburi  3127.429462



## 🔹 แบบฝึกหัด 3: ใช้ GeoPandas ทำ Spatial Join ระหว่างข้อมูลจังหวัดและอำเภอ


In [10]:

# โหลดข้อมูลชั้นข้อมูลอำเภอ
gdf_districts = gpd.read_file("/content/drive/MyDrive/Colab Notebooks/ไืทย/gadm41_THA_2.json")

# ทำ Spatial Join ระหว่างอำเภอกับจังหวัด
gdf_joined = gpd.sjoin(gdf_districts, gdf, how="inner", predicate="within")

# แสดงตัวอย่างข้อมูลที่เชื่อมโยงกัน
print(gdf_joined.head())


        GID_2 GID_0_left COUNTRY_left GID_1_left        NAME_1_left  \
15  THA.3.2_1        THA     Thailand    THA.3_1  BangkokMetropolis   
16  THA.3.3_1        THA     Thailand    THA.3_1  BangkokMetropolis   
17  THA.3.4_1        THA     Thailand    THA.3_1  BangkokMetropolis   
18  THA.3.5_1        THA     Thailand    THA.3_1  BangkokMetropolis   
21  THA.3.8_1        THA     Thailand    THA.3_1  BangkokMetropolis   

      NL_NAME_1_left       NAME_2 VARNAME_2  NL_NAME_2 TYPE_2  ...  \
15  จังหวัดเชียงใหม่     BangKapi        NA     บางกะป   Khet  ...   
16  จังหวัดเชียงใหม่     BangKhae        NA      บางแค   Khet  ...   
17  จังหวัดเชียงใหม่     BangKhen        NA     บางเขน   Khet  ...   
18  จังหวัดเชียงใหม่  BangKhoLaem        NA  บางคอแหลม   Khet  ...   
21  จังหวัดเชียงใหม่      BangRak        NA     บางรัก   Khet  ...   

   GID_0_right COUNTRY_right        NAME_1_right  \
15         THA      Thailand  Bangkok Metropolis   
16         THA      Thailand  Bangkok Metropolis


## 🔹 แบบฝึกหัด 4: ใช้ NumPy และ GeoPandas ร่วมกันเพื่อหาข้อมูลจังหวัดที่มี NDVI เฉลี่ยสูงสุด


In [11]:
# สร้างข้อมูล NDVI เฉลี่ยแบบสุ่มสำหรับแต่ละจังหวัด (ค่าระหว่าง 0.1 ถึง 0.9 ซึ่งเป็นค่า NDVI ทั่วไปสำหรับพืชพรรณ)
gdf["NDVI_average"] = np.random.uniform(low=0.1, high=0.9, size=len(gdf))

# หาจังหวัดที่มี NDVI เฉลี่ยสูงสุด
province_highest_ndvi = gdf.nlargest(1, "NDVI_average")[["NAME_1", "NDVI_average"]]

print("จังหวัดที่มี NDVI เฉลี่ยสูงสุด:")
print(province_highest_ndvi)


จังหวัดที่มี NDVI เฉลี่ยสูงสุด:
       NAME_1  NDVI_average
64  Sukhothai      0.897641


6606614714 ธนัชชา สันติเตชกุล